In [1]:
%pwd

'/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI/research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI'

In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/cg/5mr1myzx7k15hkdc69d4bh_h0000gn/T/ipykernel_76645/3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI/wellnessai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Extract data from pdf file

In [5]:
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob= '*.pdf', loader_cls= PyPDFLoader)
    documents = loader.load()
    return documents

In [6]:
import os
print(os.getcwd())

/Users/niteshkr.jha/Desktop/Agentic AI Projects/WellnessAI


In [7]:
extracted_data = load_pdf_file("data")

In [8]:
print(len(extracted_data))

4505


In [9]:
print(extracted_data[0])

page_content='' metadata={'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-16T20:19:33+02:00', 'moddate': '2006-10-16T22:03:45+02:00', 'source': 'data/GEM_3rd_edition.pdf', 'total_pages': 4505, 'page': 0, 'page_label': 'i'}


## Chunking Document Objects

In [10]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [11]:
text_chunks = text_split(extracted_data=extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 40059


In [12]:
text_chunks[0]

Document(metadata={'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'creator': 'Adobe Acrobat 6.0', 'creationdate': '2006-10-16T20:19:33+02:00', 'moddate': '2006-10-16T22:03:45+02:00', 'source': 'data/GEM_3rd_edition.pdf', 'total_pages': 4505, 'page': 1, 'page_label': 'ii'}, page_content='The GALE\nENCYCLOPEDIA of\nMEDICINE\nTHIRD EDITION')

## Download the Embeddings from Hugging Face

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [14]:
embeddings_model = download_hugging_face_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8224.13it/s]


In [15]:
query_result = embeddings_model.embed_query("Hello World")
print("length", len(query_result))

length 384


## Setting Up Pinecone DB

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()  ## important for using .env file 

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
print(PINECONE_API_KEY)

pcsk_471MPp_8wZYqBr5Srpi1wrgZt8g8gr3avaS96GeNowkzH8xD1wQiSbaH8giZQAMsLYXZu2


In [25]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key= PINECONE_API_KEY)

index_name = "wellnessai"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        vector_type="dense",
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
        deletion_protection="disabled",
        tags={
            "environment": "development"
        }
    )

### Embed each chunk and upsert the embeddings into our pinecone index

In [26]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding = embeddings_model
)

In [27]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs = {'k':3}) # gives 3 relevent answer

In [28]:
retrieved_docs = retriever.invoke("What is Acne?")
print(retrieved_docs)

[Document(id='23ca924c-cf2e-4b0a-8067-c31878c2af13', metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2006-10-16T22:03:45+02:00', 'page': 55.0, 'page_label': '26', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'data/GEM_3rd_edition.pdf', 'total_pages': 4505.0}, page_content='Researchers, Inc. Reproduced by permission.)\n26 GALE ENCYCLOPEDIA OF MEDICINE\nAcne'), Document(id='0c402ce6-2043-482d-811a-d5428cc1b5bf', metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2006-10-16T22:03:45+02:00', 'page': 269.0, 'page_label': '240', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'data/GEM_3rd_edition.pdf', 'total_pages': 4505.0}, page_content='forms of acne.\nPurpose\nDifferent types of antiacne drugs are used for\ndifferent purposes. For example, lotions, soaps, gels,\nand creams containing benzoyl peroxide or tretinoin\nmay be used to clear up mild to moderately severe\nacne. Isotretinoin

### Initializing Cloud Model

In [ ]:
import os
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv()
# Ensure your API key is configured
OLLAMA_API_KEY = os.environ["OLLAMA_API_KEY"]

# Initialize the model using Ollama's Cloud endpoint
llm = ChatOllama(
    base_url="https://ollama.com",
    model="minimax-m2.5:cloud", # Alternative free models: "kimi-k2.5:cloud" or "glm-5:cloud"
    temperature=0.7
)

# Invoke the model
print("Sending request...")

response = llm.invoke("what is ai in 100 words")

print(response.content)


Sending request...
Artificial intelligence (AI) refers to computer systems designed to mimic human cognition, enabling them to learn, reason, and solve problems. AI encompasses techniques such as machine learning, where algorithms improve through data, and deep learning, which uses neural networks with many layers to handle complex patterns. Applications range from voice assistants and image recognition to autonomous vehicles and medical diagnosis. While AI can boost efficiency and unlock new possibilities, it raises ethical concerns about bias, privacy, and job displacement. Ongoing research aims to create more robust, interpretable, and trustworthy AI, ensuring it augments human capabilities rather than replaces them today.


## Connect cloud llm to our langchain framework system

In [41]:
# Change 'from langchain.chains' to 'from langchain_classic.chains'
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Core components remain in langchain_core
from langchain_core.prompts import ChatPromptTemplate

# system_prompt = (
#     "You are an assistant for question-answering tasks. "
#     "Use the following pieces of retrieved context to answer "
#     "the question. If you don't know the answer, say that you "
#     "don't know. Use three sentences maximum and keep the "
#     "answer concise."
#     "\n\n"
#     "{context}"
# )
system_prompt = (
    "You are an empathetic, professional Wellness AI assistant. "
    "Your goal is to guide users safely and provide structured health information.\n\n"
    "Use the following retrieved context to answer the user's question. "
    "If the retrieved context does not contain enough information, say you don't know rather than making up information.\n\n"
    "Retrieved Context:\n"
    "{context}\n\n"
    "Before answering, internally reason through the following steps. "
    "Do not reveal your internal reasoning or thought process in your response.\n\n"
    "Reasoning protocol:\n"
    "1. Determine whether the user's query is related to health, medicine, or wellness.\n"
    "2. Classify the query as one of:\n"
    "   - General health knowledge\n"
    "   - Personal symptom or medical guidance request\n"
    "   - Out-of-context\n"
    "3. If the user describes symptoms, determine whether sufficient information is available.\n"
    "4. Choose the appropriate response format below.\n\n"
    "Response Rules:\n\n"
    "1. Out-of-context query:\n"
    'Respond only with:\n'
    '"I am a wellness AI. I cannot assist with out-of-context tasks."\n\n'
    "2. General knowledge query:\n"
    "- Provide a concise educational answer.\n"
    "- Do not ask follow-up questions.\n"
    "- Do not include wellness tips.\n\n"
    "3. Symptom query with insufficient information:\n"
    "- Ask one or two polite, targeted follow-up questions to better understand the symptoms.\n"
    "- Do not diagnose.\n\n"
    "4. Symptom query with sufficient information:\n"
    "Respond exactly in the following format:\n\n"
    "Diseases Asked: <possible disease or condition>\n"
    "Symptoms You Have:\n"
    "- <symptom 1>\n"
    "- <symptom 2>\n\n"
    "Wellness Tip:\n"
    "<Provide practical, safe lifestyle advice and include a brief disclaimer encouraging consultation with a qualified healthcare professional.>\n\n"
    "Keep responses concise, accurate, empathetic, and based only on the retrieved context whenever possible."
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [42]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [43]:
response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])

Acne is a common skin condition that occurs when pores or hair follicles become blocked. This blockage allows a waxy material called sebum—an oily substance produced by sebaceous glands—to collect inside the pores. When the sebaceous glands become inflamed due to this buildup, it results in what is commonly known as acne or acne vulgaris.

The sebaceous follicles, which house the oil-producing glands and hair follicles, are where pimples form. Acne can range from mild to severe and may present as blackheads, whiteheads, papules, pustules, or cysts.
